# Autonomous AI Analyst — End-to-End Fraud Detection Pipeline

**Dataset:** Kaggle Credit Card Fraud Detection · 284,807 transactions, 0.173% fraud rate (severe class imbalance).

This notebook runs the deterministic data-science pipeline from the [autonomous-ai-analyst repo](https://github.com/Saqibnazirbhat/autonomous-ai-analyst) end to end:

1. **Data cleaning** — schema validation, dtype coercion, null check
2. **Exploratory analysis** — class balance, top correlated features, transaction amount distribution
3. **Model training** — XGBoost with `scale_pos_weight` to handle the 1:578 class imbalance, stratified 80/20 split
4. **Confusion matrix** — TN/FP/FN/TP with discussion of the false-negative cost
5. **Robustness check** — F1 across 20 random seeds with bootstrap CI
6. **Drift check** — Kolmogorov-Smirnov test along the Time-median split
7. **ROI / cost analysis** — expected value at FN=$500 / FP=$5 vs. flag-nothing baseline

**Target:** F1 ≥ 0.85 on the held-out fraud class. The repo's full pipeline also runs a four-agent LLM debate (bull / bear / devil / judge) on top of these numbers, but that lives outside the notebook.


## 1. Setup

Detect dataset path (Kaggle vs. local), import dependencies, fix the global random seed.


In [ ]:
import os
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ks_2samp
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
)
from xgboost import XGBClassifier

RNG_SEED = 42
np.random.seed(RNG_SEED)

# Locate the dataset: Kaggle convention first, fall back to common alternatives.
CANDIDATES = [
    "/kaggle/input/creditcardfraud/creditcard.csv",
    "../input/creditcardfraud/creditcard.csv",
    "data/creditcard.csv/creditcard.csv",
    "data/creditcard.csv",
]
DATA_PATH = next((p for p in CANDIDATES if os.path.exists(p)), None)
assert DATA_PATH is not None, f"Dataset not found in any of: {CANDIDATES}"
print(f"Using dataset: {DATA_PATH}")

sns.set_context("notebook")


## 2. Data Cleaning

The Kaggle CSV is documented as having zero nulls and a fixed schema. Strategy here is:

- **Assert** the expected shape and column set (fail fast if upstream changed)
- **Verify** zero nulls (do not impute silently — a null would indicate upstream corruption)
- **Coerce** dtypes: `Time`, `V1..V28`, `Amount` → `float64`; `Class` → `int8`
- **Save** a clean copy as parquet for downstream reuse


In [ ]:
df = pd.read_csv(DATA_PATH)

EXPECTED_COLS = ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount", "Class"]
assert df.shape == (284807, 31), f"shape mismatch: {df.shape}"
assert list(df.columns) == EXPECTED_COLS, f"columns mismatch: {list(df.columns)}"
assert df.isnull().sum().sum() == 0, "null values detected — investigate upstream"

for col in ["Time"] + [f"V{i}" for i in range(1, 29)] + ["Amount"]:
    df[col] = df[col].astype("float64")
df["Class"] = df["Class"].astype("int8")

assert set(df["Class"].unique()) == {0, 1}

print(f"rows: {len(df):,}")
print(f"columns: {len(df.columns)}")
print(f"nulls: {int(df.isnull().sum().sum())}")
print(f"dtypes: {df.dtypes.value_counts().to_dict()}")

OUT_DIR = Path("/kaggle/working") if os.path.isdir("/kaggle/working") else Path(".")
OUT_DIR.mkdir(parents=True, exist_ok=True)
clean_path = OUT_DIR / "clean.parquet"
df.to_parquet(clean_path, compression="snappy", index=False)
print(f"\nClean parquet -> {clean_path} ({clean_path.stat().st_size / 1e6:.1f} MB)")


## 3. Exploratory Data Analysis

Three things that matter for a fraud model:

1. **Class balance** — how skewed is the positive class?
2. **Feature signal** — which `V*` features carry usable correlation with the target?
3. **Amount distribution** — fraud amounts often differ in tail behavior from legitimate ones


### 3.1 Class balance


In [ ]:
counts = df["Class"].value_counts().sort_index()
fraud_rate = float(counts[1] / counts.sum())
imbalance_ratio = counts[0] / counts[1]

print(f"legitimate (0): {counts[0]:,}")
print(f"fraud      (1): {counts[1]:,}")
print(f"fraud rate    : {fraud_rate:.6f}  ({fraud_rate * 100:.4f}%)")
print(f"imbalance     : 1 fraud per {imbalance_ratio:,.1f} legitimate transactions")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(["legitimate (0)", "fraud (1)"], counts.values, color=["#4C72B0", "#C44E52"])
ax.set_yscale("log")
ax.set_ylabel("count (log scale)")
ax.set_title("Class balance")
for i, v in enumerate(counts.values):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom")
plt.tight_layout()
plt.show()


### 3.2 Top features by absolute correlation with `Class`

`V1..V28` are PCA-transformed numeric features (Kaggle does not publish the original variable names). Even modest correlations are useful when combined in a tree ensemble.


In [ ]:
TARGET = "Class"
FEATURES = [f"V{i}" for i in range(1, 29)] + ["Amount", "Time"]

corr = df[FEATURES + [TARGET]].corr()[TARGET].drop(TARGET)
top10 = corr.abs().sort_values(ascending=False).head(10)

print("Top 10 features by |corr with Class|:")
for name, val in top10.items():
    print(f"  {name:6s}  {val:.4f}")

fig, ax = plt.subplots(figsize=(7, 5))
ax.barh(top10.index[::-1], top10.values[::-1], color="#55A868")
ax.set_xlabel("|correlation with Class|")
ax.set_title("Top 10 features by absolute correlation")
plt.tight_layout()
plt.show()


### 3.3 Transaction amount by class (symlog scale handles the heavy tails)


In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
ax.boxplot(
    [df.loc[df.Class == 0, "Amount"].values,
     df.loc[df.Class == 1, "Amount"].values],
    tick_labels=["legitimate", "fraud"],
    showfliers=True,
)
ax.set_yscale("symlog")
ax.set_ylabel("Amount (symlog)")
ax.set_title("Transaction Amount by Class")
plt.tight_layout()
plt.show()

print("\nAmount stats by class:")
print(df.groupby("Class")["Amount"].agg(["mean", "median", "std", "max"]).round(2))


## 4. Model Training

**Algorithm choice: XGBoost with `scale_pos_weight`.** Tree-based models handle the mixed-scale features (raw `Time` and `Amount` plus zero-centered PCA components `V1..V28`) without standardization. `scale_pos_weight ≈ 578` corrects for the class imbalance during training while preserving the true class distribution at inference.

**Split: stratified 80/20.** At a 0.17% fraud rate a non-stratified split would risk producing a test fold with too few positives to estimate F1 reliably.

**Acceptance gate: F1 ≥ 0.85** on the fraud class on the held-out test set.


In [ ]:
y = df[TARGET].astype(int)
X = df.drop(columns=[TARGET])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RNG_SEED
)
neg = int((y_train == 0).sum())
pos = int((y_train == 1).sum())
scale_pos_weight = neg / pos

print(f"train rows: {len(X_train):,}    test rows: {len(X_test):,}")
print(f"train pos: {pos:,}    train neg: {neg:,}")
print(f"scale_pos_weight: {scale_pos_weight:.4f}")

hyperparams = dict(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.1,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    random_state=RNG_SEED,
    tree_method="hist",
    n_jobs=-1,
)
model = XGBClassifier(**hyperparams)
model.fit(X_train, y_train)
print("\nTraining complete.")


### 4.1 Held-out test set evaluation


In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

metrics = {
    "f1":        float(f1_score(y_test, y_pred, pos_label=1)),
    "precision": float(precision_score(y_test, y_pred, pos_label=1)),
    "recall":    float(recall_score(y_test, y_pred, pos_label=1)),
    "auc":       float(roc_auc_score(y_test, y_proba)),
}

print("Test-set metrics (fraud class, positive=1):")
for k, v in metrics.items():
    print(f"  {k:10s}  {v:.4f}")

passed = metrics["f1"] >= 0.85
print(f"\nF1 >= 0.85 acceptance gate: {'PASS' if passed else 'FAIL'}")
assert passed, f"F1 {metrics['f1']:.4f} below 0.85 target"


## 5. Confusion Matrix

The confusion matrix is the most decision-relevant view: how many fraud transactions slipped through (FN), how many legitimate transactions were unfairly flagged (FP).


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
tn, fp = int(cm[0, 0]), int(cm[0, 1])
fn, tp = int(cm[1, 0]), int(cm[1, 1])
total = tn + fp + fn + tp

print("               predicted 0     predicted 1")
print(f"actual 0   {tn:>12,}    {fp:>12,}")
print(f"actual 1   {fn:>12,}    {tp:>12,}")
print()
print(f"True negatives  TN = {tn:,}    ({tn/total*100:.4f}% of test)")
print(f"False positives FP = {fp:,}    ({fp/(tn+fp)*100:.4f}% of legit transactions)")
print(f"False negatives FN = {fn:,}    ({fn/(fn+tp)*100:.4f}% of fraud — missed frauds cost real money)")
print(f"True positives  TP = {tp:,}    ({tp/(fn+tp)*100:.4f}% of fraud — recall on the positive class)")

fig, ax = plt.subplots(figsize=(5.5, 4))
sns.heatmap(
    cm, annot=True, fmt=",", cmap="Blues",
    xticklabels=["predicted 0", "predicted 1"],
    yticklabels=["actual 0", "actual 1"], ax=ax,
    cbar=False, annot_kws={"fontsize": 13},
)
ax.set_title("Confusion matrix (test set)")
plt.tight_layout()
plt.show()


## 6. Robustness Check — F1 Distribution Across 20 Seeds

A single F1 number tells you nothing about how stable that number is. With only 98 fraud samples in the test slice, sampling noise alone can move F1 by 5+ percentage points.

We refit XGBoost (same hyperparameters, same stratified 80/20 split design) across 20 different random seeds — each seed produces a different train/test partition AND a different XGBoost initialization. The spread of F1 values is the real uncertainty estimate.


In [ ]:
N_SEEDS = 20  # ~5-7 seconds per fit on Kaggle CPU; total ~2 minutes
seed_f1s = []

print(f"Fitting XGBoost across {N_SEEDS} seeds (this takes ~2 min)...")
for s in range(N_SEEDS):
    Xs_tr, Xs_te, ys_tr, ys_te = train_test_split(
        X, y, test_size=0.2, stratify=y, random_state=s
    )
    spw = (ys_tr == 0).sum() / (ys_tr == 1).sum()
    m = XGBClassifier(
        n_estimators=200, max_depth=6, learning_rate=0.1,
        scale_pos_weight=spw, eval_metric="logloss",
        random_state=s, tree_method="hist", n_jobs=-1,
    )
    m.fit(Xs_tr, ys_tr)
    f1 = float(f1_score(ys_te, m.predict(Xs_te), pos_label=1))
    seed_f1s.append(f1)
    print(f"  seed {s:2d}: F1 = {f1:.4f}")

f1_arr = np.array(seed_f1s)
above_bar = int((f1_arr >= 0.85).sum())

rng = np.random.default_rng(0)
boot_means = [rng.choice(f1_arr, size=len(f1_arr), replace=True).mean() for _ in range(2000)]
ci_lo, ci_hi = np.percentile(boot_means, [2.5, 97.5])

print(f"\nF1 mean : {f1_arr.mean():.4f}")
print(f"F1 std  : {f1_arr.std(ddof=1):.4f}")
print(f"F1 min  : {f1_arr.min():.4f}")
print(f"F1 max  : {f1_arr.max():.4f}")
print(f"95% bootstrap CI on mean: [{ci_lo:.4f}, {ci_hi:.4f}]")
print(f"Seeds clearing 0.85 bar: {above_bar} / {N_SEEDS}")


In [ ]:
sorted_f1s = np.sort(f1_arr)
colors = ["#C44E52" if v < 0.85 else "#4C72B0" for v in sorted_f1s]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(len(sorted_f1s)), sorted_f1s, color=colors, edgecolor="none")
ax.axhline(0.85, color="#FBBF24", linestyle="--", linewidth=1.5, label="F1 = 0.85 acceptance bar")
ax.axhline(f1_arr.mean(), color="#55A868", linestyle=":", linewidth=1.5,
           label=f"mean = {f1_arr.mean():.4f}")
ax.set_ylim(min(0.80, sorted_f1s.min() - 0.01), max(0.92, sorted_f1s.max() + 0.01))
ax.set_xlabel("seed (sorted by F1)")
ax.set_ylabel("F1 (fraud class)")
ax.set_title(f"F1 distribution across {N_SEEDS} stratified 80/20 splits")
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()


## 7. Drift Check — Kolmogorov-Smirnov Test on Time-Median Split

The dataset spans roughly two days. If we treat the first half (by `Time`) as a baseline batch and the second half as a new batch, are their feature distributions actually the same?

Naive `rel_delta` metrics (relative change in feature mean) are unreliable on PCA-derived features whose population mean is engineered to be close to zero — the denominator vanishes and the ratio saturates.

**KS test** is the right tool: it compares the full empirical distributions, not just means. We report the count of features that reject the null hypothesis of identical distributions at p < 0.05.


In [ ]:
t_median = df["Time"].median()
baseline = df[df["Time"] <= t_median]
new_batch = df[df["Time"] > t_median]
drift_features = [f"V{i}" for i in range(1, 29)] + ["Amount"]

print(f"baseline rows: {len(baseline):,}    new batch rows: {len(new_batch):,}")
print(f"baseline fraud rate: {baseline['Class'].mean():.6f}")
print(f"new batch fraud rate: {new_batch['Class'].mean():.6f}")

ks_results = []
for col in drift_features:
    stat, p = ks_2samp(baseline[col].values, new_batch[col].values)
    ks_results.append((col, float(stat), float(p)))

n_significant = sum(1 for _, _, p in ks_results if p < 0.05)
print(f"\nKS-significant features (p < 0.05): {n_significant} / {len(drift_features)}")

print("\nTop 10 drifted features by KS statistic:")
for col, stat, p in sorted(ks_results, key=lambda x: x[1], reverse=True)[:10]:
    p_repr = f"{p:.2e}" if p > 1e-300 else "< 1e-300"
    print(f"  {col:7s}  KS = {stat:.4f}  p = {p_repr}")


In [ ]:
shuffled = df.sample(frac=1.0, random_state=RNG_SEED).reset_index(drop=True)
half = len(shuffled) // 2
ctrl_baseline = shuffled.iloc[:half]
ctrl_new = shuffled.iloc[half:]

ctrl_significant = sum(
    1 for col in drift_features
    if ks_2samp(ctrl_baseline[col].values, ctrl_new[col].values)[1] < 0.05
)
print(f"Random-shuffle control: {ctrl_significant} / {len(drift_features)} features KS-significant")
print("(Should be near 0 if there is no real drift introduced by random splitting.)")


## 8. ROI / Cost Analysis

F1 is a balanced metric, but real deployments care about **expected dollar value**. The two error types have very different costs:

- **False negative (FN)** — a real fraud the model missed. Cost: the full transaction amount, conservatively averaged at **$500** per missed fraud.
- **False positive (FP)** — a legitimate transaction unnecessarily reviewed. Cost: ~5 minutes of analyst time at $1/min = **$5** per flag.

That **100:1 cost ratio** is the load-bearing assumption. Under it, even a recall gap that looks small in F1 terms can shift expected value materially.

We compare three policies on the held-out test slice:

| Policy | Cost formula |
|---|---|
| **Model** | `FN × $500 + FP × $5` |
| **Flag nothing** | `(FN + TP) × $500` (every fraud passes through) |
| **Flag everything** | `(TN + FP) × $5` (every legit transaction reviewed) |


In [ ]:
FN_COST = 500.0
FP_COST = 5.0

model_cost     = fn * FN_COST + fp * FP_COST
flag_none_cost = (fn + tp) * FN_COST
flag_all_cost  = (tn + fp) * FP_COST

savings_vs_none = flag_none_cost - model_cost
savings_vs_all  = flag_all_cost - model_cost

time_span_sec = float(df["Time"].max() - df["Time"].min())
test_days = (time_span_sec / 86400.0) * 0.2
savings_per_day = savings_vs_none / test_days

print(f"Cost matrix:  FN = ${FN_COST:.0f}    FP = ${FP_COST:.0f}    ratio = {FN_COST / FP_COST:.0f}:1")
print(f"Test slice covers ~{test_days:.2f} days of transaction volume")
print()
print(f"{'policy':<20}  {'cost':>14}")
print(f"{'-'*20}  {'-'*14}")
print(f"{'model':<20}  ${model_cost:>13,.2f}")
print(f"{'flag nothing':<20}  ${flag_none_cost:>13,.2f}")
print(f"{'flag everything':<20}  ${flag_all_cost:>13,.2f}")
print()
print(f"Savings vs. flag-nothing : ${savings_vs_none:>13,.2f}  per test slice")
print(f"Savings vs. flag-all     : ${savings_vs_all:>13,.2f}  per test slice")
print(f"Daily extrapolation      : ${savings_per_day:>13,.2f}  per day")


In [ ]:
policies = ["model", "flag nothing", "flag everything"]
costs = [model_cost, flag_none_cost, flag_all_cost]
colors = ["#55A868", "#C44E52", "#FBBF24"]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(policies, costs, color=colors, edgecolor="none")
ax.set_ylabel("cost (USD, test slice)")
ax.set_title("Policy cost comparison  (FN cost $500, FP cost $5)")
for bar, c in zip(bars, costs):
    ax.text(bar.get_x() + bar.get_width() / 2, c, f"${c:,.0f}",
            ha="center", va="bottom", fontsize=11)
ax.set_ylim(0, max(costs) * 1.15)
plt.tight_layout()
plt.show()


## 9. Summary

| Result | Value |
|---|---|
| Test-set F1 | shown above (target: ≥ 0.85) |
| Test-set AUC | shown above |
| 20-seed F1 mean | shown above |
| 95% bootstrap CI on F1 mean | shown above (typically [0.855, 0.876]) |
| Seeds clearing 0.85 | shown above |
| KS-significant features (Time split) | 29 / 29 — real Time-axis drift exists |
| KS-significant features (random-shuffle control) | 0 / 29 — confirms KS calibration |
| Modeled savings vs. flag-nothing | shown above |
| Daily savings extrapolation | shown above |

**Headline finding:** the model clears the F1 ≥ 0.85 acceptance bar both at the headline number and at the 95% CI lower bound across 20 seeds. Under a 100:1 FN:FP cost matrix, expected dollar value strongly favors deployment with analyst-in-loop review on the uncertain band, even acknowledging the genuine Time-axis drift the KS test surfaces.

**What this notebook deliberately omits:**

- The four-agent LLM debate (`bull`, `bear`, `devil`, `judge`) that this repo runs on top of these numbers — the verdict ends with the judge writing a non-empty `dissent_note` recording the strongest counter-view, even when the agents converge.
- Calibration analysis (Brier, Platt, isotonic) on `predict_proba` — for cost-aware threshold tuning at `p* = FP_cost / (FP_cost + FN_cost) ≈ 0.0099`.
- Concept drift on `P(Y|X)` rather than just covariate drift on `P(X)` — only the latter is testable here without label leakage.

Full pipeline, agent contracts, governance trace, and dashboard live at: <https://github.com/Saqibnazirbhat/autonomous-ai-analyst>
